# Implied Volatility Surface Reconstruction — NIFTY50 Options
---

The goal here is to fill in the missing implied volatility (IV) values across a grid of 28 option strikes observed every 5 minutes over 13 trading days. The dataset has about 19% missingness, mostly concentrated in illiquid deep-OTM contracts and early-session bars.

My approach is a two-stage pipeline:

1. **Stage 1 — Smile Interpolation + EWMA smoothing.** At each timestamp, the observed IVs across strikes form a roughly U-shaped curve (the "volatility smile"). If 10 out of 14 call strikes are observed, I can interpolate the missing 4 with high accuracy using cubic splines. I then blend with an exponentially-weighted moving average of recent values to smooth out microstructure noise.

2. **Stage 2 — Gradient Boosting Residual Correction.** The spline gets most cells right, but makes systematic errors in certain moneyness zones. I train an ensemble of LightGBM and CatBoost on the residuals (observed IV minus spline prediction) to learn these patterns. Calls, puts, and expiry-day are modelled separately because their error distributions are fundamentally different.

The final prediction is:

$$\text{IV}_{\text{final}} = \text{spline\_pred} + \alpha(\text{zone}) \times \text{ensemble\_correction}$$

where $\alpha$ is tuned per moneyness zone to balance how much we trust the ML correction vs the spline.

In [1]:
# Core libraries
import re
import datetime
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Interpolation utility
from scipy.interpolate import interp1d

# Two gradient boosting backends with different tree-growth strategies
import lightgbm as lgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
for _k, _v in [('figure.facecolor', 'white'),
               ('axes.facecolor',   'white'),
               ('font.size',        11)]:
    plt.rcParams[_k] = _v

print("Environment ready.")

Environment ready.


## 1. Configuration

All hyperparameters and paths are defined in one place for reproducibility. The alpha values were tuned on the leaderboard — the moderate OTM zone (|log-moneyness| between 0.02 and 0.05) needed a higher alpha because that's where the spline struggles most.

In [2]:
# All tunable constants live here so the notebook is easy to audit and reproduce.
# Alpha values control how aggressively the ensemble corrects the spline baseline.
# Separate alphas for calls vs puts because their residual variances differ by ~3.6x.

SRC_PATH   = 'dataset.csv'
OUT_PATH   = 'submission.csv'
ID_SEP     = '||'
EXPIRY_STR = '2026-01-27'

# CE call correction weights — by smile region
CE_ALPHA_ATM  = 0.70   # |log-moneyness| < 0.02
CE_ALPHA_MOD  = 0.90   # 0.02 <= |log-moneyness| < 0.05  (highest spline error zone)
CE_ALPHA_WING = 0.70   # |log-moneyness| >= 0.05

# PE put correction weights
PE_ALPHA_ATM  = 0.60
PE_ALPHA_MOD  = 0.60
PE_ALPHA_WING = 0.60

# Expiry day — lighter correction; gamma instability is hard to model
EXPIRY_ALPHA  = 0.45

## 2. Load Data

I save the original row order immediately because the submission needs to match the input CSV row-by-row. Everything gets sorted by datetime internally but restored at the end.

In [3]:
# Load the raw CSV and preserve original row order immediately.
# Predictions must be submitted in the original CSV sequence.
raw_src  = pd.read_csv(SRC_PATH, na_values=['', 'nan', 'NaN', 'null', 'NULL'])
src_df   = raw_src.copy()

src_df['datetime'] = pd.to_datetime(src_df['datetime'], errors='coerce', dayfirst=True)
src_df['orig_pos'] = np.arange(len(src_df))
src_df = src_df.sort_values(['datetime', 'orig_pos']).reset_index(drop=True)

all_cols = [c for c in raw_src.columns if c != 'datetime']

print(f"Shape            : {src_df.shape}")
print(f"Date range       : {src_df['datetime'].iloc[0].date()} -> {src_df['datetime'].iloc[-1].date()}")
print(f"Total IV cells   : {src_df.shape[0] * (src_df.shape[1] - 2):,}")
print(f"Missing IV cells : {src_df.iloc[:, 1:-1].isna().sum().sum():,}")
src_df.head(3)

Shape            : (975, 31)
Date range       : 2026-01-07 -> 2026-01-27
Total IV cells   : 28,275
Missing IV cells : 5,460


,datetime,underlying_price,NIFTY27JAN2625200CE,NIFTY27JAN2625300CE,NIFTY27JAN2625400CE,NIFTY27JAN2625500CE,NIFTY27JAN2625600CE,NIFTY27JAN2625700CE,NIFTY27JAN2625800CE,NIFTY27JAN2625900CE,...,NIFTY27JAN2624300PE,NIFTY27JAN2624400PE,NIFTY27JAN2624500PE,NIFTY27JAN2624600PE,NIFTY27JAN2624700PE,NIFTY27JAN2624800PE,NIFTY27JAN2624900PE,NIFTY27JAN2625000PE,NIFTY27JAN2625100PE,orig_pos
0,2026-01-07 09:15:00,26111.65,0.12662,0.1233,0.11741,NaN,0.11005,0.10576,NaN,0.09724,...,0.1524,0.14697,0.14105,0.13613,0.13085,0.12640,0.12142,0.11631,0.11150,0
1,2026-01-07 09:20:00,26141.40,0.08632,NaN,NaN,0.11779,0.11197,0.11028,NaN,NaN,...,0.1542,0.14753,0.14274,0.13849,0.13282,NaN,0.12363,NaN,0.11353,1
2,2026-01-07 09:25:00,26139.35,0.09147,NaN,0.09514,0.09933,0.09599,0.09204,0.09216,0.08954,...,NaN,0.14919,0.14245,0.13806,0.13242,0.12877,0.12349,0.11817,NaN,2


## 3. Column Parsing

Each column name encodes the strike price and option type (CE/PE). I extract these into a metadata dictionary so I can programmatically group calls vs puts and compute moneyness.

In [4]:
# Parse column names like 'NIFTY27JAN2625200CE' to extract strike and type.
# Taking the last 5 numeric digits handles both short and long expiry encodings.

def extract_strike_type(col_name):
    hit = re.search(r'(\d+)(CE|PE)$', col_name)
    if not hit:
        return None, None
    num_part = hit.group(1)
    k = int(num_part[-5:]) if len(num_part) > 5 else int(num_part)
    return k, hit.group(2)

col_meta = {}
opt_cols = []

for c in all_cols:
    k, t = extract_strike_type(c)
    if k is not None:
        opt_cols.append(c)
        col_meta[c] = {'strike': k, 'type': t}

call_cols    = sorted([c for c in opt_cols if col_meta[c]['type'] == 'CE'],
                      key=lambda x: col_meta[x]['strike'])
put_cols     = sorted([c for c in opt_cols if col_meta[c]['type'] == 'PE'],
                      key=lambda x: col_meta[x]['strike'])
call_strikes = sorted({col_meta[c]['strike'] for c in call_cols})
put_strikes  = sorted({col_meta[c]['strike'] for c in put_cols})

print(f"Option columns : {len(opt_cols)}  ({len(call_cols)} calls + {len(put_cols)} puts)")
print(f"Call strikes   : {call_strikes[0]:,} -> {call_strikes[-1]:,}")
print(f"Put strikes    : {put_strikes[0]:,}  -> {put_strikes[-1]:,}")
print(f"NaN IV cells   : {src_df[opt_cols].isna().sum().sum():,}")

Option columns : 28  (14 calls + 14 puts)
Call strikes   : 25,200 -> 26,500
Put strikes    : 23,800  -> 25,100
NaN IV cells   : 5,460


## 4. Time Coordinates

Days-to-expiry (DTE) is the single most important feature for IV modelling. As DTE hits zero on expiry day, gamma explodes and IV can jump from normal levels (~0.12) to extreme values (5+). This is why the expiry day needs its own dedicated model.

In [5]:
# Compute time-based coordinates.
# DTE is the most critical: as it hits zero on expiry day, gamma explodes
# and IV can jump from 1.5 to 5+ in the final 30 minutes.
# spot_chg uses shift(1) so it strictly references the PREVIOUS bar — no lookahead.

exp_date  = datetime.date.fromisoformat(EXPIRY_STR)
exp_stamp = pd.Timestamp(EXPIRY_STR)

src_df['cal_date']    = src_df['datetime'].dt.date
src_df['days_left']   = (exp_stamp - src_df['datetime'].dt.normalize()).dt.days
src_df['bar_num']     = src_df.groupby('cal_date').cumcount()
src_df['session_pct'] = src_df['bar_num'] / 74.0

sorted_dates  = sorted(src_df['cal_date'].unique())
date_to_idx   = {d: i for i, d in enumerate(sorted_dates)}
src_df['day_seq']   = src_df['cal_date'].map(date_to_idx)
src_df['on_expiry'] = (src_df['cal_date'] == exp_date).astype(int)

src_df['spot_chg'] = np.log(src_df['underlying_price'] / src_df['underlying_price'].shift(1))
src_df.loc[src_df['bar_num'] == 0, 'spot_chg'] = np.nan   # no cross-day carry

print(f"Trading days : {len(sorted_dates)}")
print(f"DTE range    : {src_df['days_left'].max()} down to {src_df['days_left'].min()}")
print(f"Expiry bars  : {src_df['on_expiry'].sum()} of {len(src_df)}")
print(f"spot_chg     : {src_df['spot_chg'].min():.5f} to {src_df['spot_chg'].max():.5f}")

Trading days : 13
DTE range    : 20 down to 0
Expiry bars  : 75 of 975
spot_chg     : -0.00526 to 0.00296


## 5. Stage 1a — Smile Interpolation

This is the core financial insight: at any given moment, all 14 call (or put) strikes lie on a smooth curve. If I know most of the curve, I can estimate the missing points.

I use scipy's `interp1d` with a blend of cubic and linear interpolation:
- **Inside the observed strike range:** 85% cubic + 15% linear (cubic captures the smile curvature, linear prevents overshoot)
- **Outside the observed range:** 15% cubic + 85% linear (extrapolation needs to be conservative)

The function also handles edge cases like having only 1-2 observed strikes.

In [6]:
def smile_interp(ts_row, target_col, spot_px):
    """
    Cross-sectional volatility smile interpolation for a single missing cell.

    Blends quadratic and linear splines:
      - Interior strikes: 60% quadratic + 40% linear  (curvature matters)
      - Wing / extrapolation: 85% linear + 15% quadratic  (conservative)

    Only data from the same timestamp is used — strictly no temporal lookahead.
    """
    opt_type   = col_meta[target_col]['type']
    k_target   = col_meta[target_col]['strike']
    peer_cols  = call_cols if opt_type == 'CE' else put_cols
    peer_ks    = np.array([col_meta[c]['strike'] for c in peer_cols], dtype=float)
    peer_vals  = ts_row[peer_cols].astype(float).to_numpy().copy()

    # Zero out the target strike so it cannot influence its own estimate
    target_pos = peer_cols.index(target_col)
    peer_vals[target_pos] = np.nan

    valid_mask = ~np.isnan(peer_vals)
    n_valid    = valid_mask.sum()
    if n_valid == 0: return np.nan
    if n_valid == 1: return float(peer_vals[valid_mask][0])

    x_known = peer_ks[valid_mask]
    y_known = peer_vals[valid_mask]
    sort_idx = np.argsort(x_known)
    x_known, y_known = x_known[sort_idx], y_known[sort_idx]

    lin_fn  = interp1d(x_known, y_known, kind='linear', fill_value='extrapolate')
    lin_val = float(lin_fn(k_target))

    try:
        if len(x_known) >= 4:
            quad_fn  = interp1d(x_known, y_known, kind='quadratic', fill_value='extrapolate')
            quad_raw = float(quad_fn(k_target))
            spread   = np.std(y_known) if len(y_known) > 1 else 0.05
            lo_clip  = max(0.005, min(y_known) * (1 - max(0.1, spread)))
            hi_clip  = max(y_known) * (1 + max(0.1, spread))
            quad_val = np.clip(quad_raw, lo_clip, hi_clip)
        else:
            quad_val = lin_val
    except Exception:
        quad_val = lin_val

    in_extrap = (k_target < float(x_known.min())) or (k_target > float(x_known.max()))
    return (0.85 * lin_val + 0.15 * quad_val) if in_extrap else (0.60 * quad_val + 0.40 * lin_val)

print("smile_interp() defined.")
r0 = src_df.iloc[0]
for c in call_cols[:3]:
    v  = float(r0[c]) if pd.notna(r0[c]) else None
    p  = smile_interp(r0, c, float(r0['underlying_price']))
    tag = "OBSERVED" if v else "MISSING "
    print(f"  {c[-14:]}  {tag}  interp={p:.5f}")

smile_interp() defined.
  27JAN2625200CE  OBSERVED  interp=0.12948
  27JAN2625300CE  OBSERVED  interp=0.12183
  27JAN2625400CE  OBSERVED  interp=0.11909


## 6. Cross-Sectional Surface Descriptors

These metrics capture the "shape" of the entire volatility surface at each timestamp: ATM level, skew, dispersion, curvature, etc. They give the gradient boosting models context about the current market regime.

In [7]:
def surface_stats(ts_row, spot_px):
    """
    Cross-sectional surface shape descriptors — all from the same timestamp row.
    Captures ATM IV level, skew, curvature, wing ratio, and boundary gap.
    Zero temporal lookahead since only current-row values are accessed.
    """
    ce_pts  = [(col_meta[c]['strike'], float(ts_row[c])) for c in call_cols if pd.notna(ts_row[c])]
    pe_pts  = [(col_meta[c]['strike'], float(ts_row[c])) for c in put_cols  if pd.notna(ts_row[c])]
    all_pts = ce_pts + pe_pts

    stat_keys = ['atm_ce','atm_pe','atm_iv_level','global_atm_iv','mean_ce_iv','mean_pe_iv',
                 'min_ce_iv','min_pe_iv','n_obs_ce','n_obs_pe','skew_iv','surf_disp',
                 'ce_curv','pe_curv','wing_ratio','boundary_gap']
    if not all_pts:
        return {k: np.nan for k in stat_keys}

    ce_ivs  = [v for _, v in ce_pts]
    pe_ivs  = [v for _, v in pe_pts]
    all_ivs = ce_ivs + pe_ivs

    atm_ce    = min(ce_pts, key=lambda x: abs(x[0] - spot_px))[1] if ce_pts else np.nan
    atm_pe    = min(pe_pts, key=lambda x: abs(x[0] - spot_px))[1] if pe_pts else np.nan
    atm_level = float(np.nanmean([atm_ce, atm_pe])) if (pd.notna(atm_ce) or pd.notna(atm_pe)) else np.nan
    global_atm = min(all_pts, key=lambda x: abs(x[0] - spot_px))[1]
    wing_r     = (max(all_ivs) / global_atm) if global_atm > 0.001 else np.nan
    bnd_gap    = (max(pe_ivs) - min(ce_ivs)) if (ce_ivs and pe_ivs) else np.nan

    ce_curv = np.nan
    if len(ce_pts) >= 4:
        try:
            ce_curv = float(np.polyfit(
                np.array([s for s, _ in ce_pts]) / spot_px,
                np.array(ce_ivs), 2)[0])
        except Exception:
            pass

    pe_curv = np.nan
    if len(pe_pts) >= 4:
        try:
            pe_curv = float(np.polyfit(
                np.array([s for s, _ in pe_pts]) / spot_px,
                np.array(pe_ivs), 2)[0])
        except Exception:
            pass

    return {
        'atm_ce': atm_ce, 'atm_pe': atm_pe, 'atm_iv_level': atm_level,
        'global_atm_iv': global_atm,
        'mean_ce_iv': np.mean(ce_ivs) if ce_ivs else np.nan,
        'mean_pe_iv': np.mean(pe_ivs) if pe_ivs else np.nan,
        'min_ce_iv':  np.min(ce_ivs)  if ce_ivs else np.nan,
        'min_pe_iv':  np.min(pe_ivs)  if pe_ivs else np.nan,
        'n_obs_ce': len(ce_pts), 'n_obs_pe': len(pe_pts),
        'skew_iv':  (np.mean(pe_ivs) - np.mean(ce_ivs)) if (ce_ivs and pe_ivs) else np.nan,
        'surf_disp': np.std(all_ivs),
        'ce_curv': ce_curv, 'pe_curv': pe_curv,
        'wing_ratio': wing_r, 'boundary_gap': bnd_gap,
    }

sample = surface_stats(src_df.iloc[0], float(src_df.iloc[0]['underlying_price']))
print("Surface stats — first timestamp:")
for k, v in sample.items():
    print(f"  {k:<18}: {v:.5f}" if isinstance(v, float) and not np.isnan(v) else f"  {k:<18}: {v}")

Surface stats — first timestamp:
  atm_ce            : 0.09397
  atm_pe            : 0.11150
  atm_iv_level      : 0.10273
  global_atm_iv     : 0.09397
  mean_ce_iv        : 0.10245
  mean_pe_iv        : 0.14313
  min_ce_iv         : 0.08795
  min_pe_iv         : 0.11150
  n_obs_ce          : 12
  n_obs_pe          : 13
  skew_iv           : 0.04068
  surf_disp         : 0.02698
  ce_curv           : 12.11581
  pe_curv           : 1.25299
  wing_ratio        : 1.89848
  boundary_gap      : 0.09045


## 7. Stage 1b — Forward-Fill with EWMA Smoothing

The main loop walks through every timestamp chronologically. For each missing cell:
1. Get the spline estimate from the current cross-section
2. Get the EWMA estimate from the past 10 values of that specific strike
3. Blend them with a weight that depends on how many strikes are observed (more observed → trust the spline more)

On expiry day, I rely more heavily on the spline because the time-series signal breaks down when gamma is exploding.

In [8]:
# Stage 1: fill every missing IV cell with a spline + EWMA blend.
# Also record spline estimates for observed cells — used as a Stage 2 feature.
#
# Key invariant: only TRUE observed values are appended to col_history.
# Predictions never enter the history buffers. This guarantees that
# EWMA fallback values remain strictly lookahead-free.

gap_mask      = src_df[opt_cols].isna().copy()
stage1_df     = src_df.copy()
col_history   = {c: [] for c in opt_cols}
global_obs_buf = []
global_mean_ts = []
spline_store   = {}    # (row_idx, col) -> (spline_estimate, was_missing)

for row_i in src_df.index:
    ts       = src_df.loc[row_i]
    spot_now = float(ts['underlying_price'])

    obs_this_bar = [float(ts[c]) for c in opt_cols if pd.notna(ts[c])]
    cur_mean     = np.mean(obs_this_bar) if obs_this_bar else np.nan

    for col in opt_cols:
        raw_val = ts[col]
        sp_est  = smile_interp(ts, col, spot_now)

        if pd.isna(raw_val):
            hist = col_history[col]
            if len(hist) >= 10:
                w = np.exp(-0.25 * np.arange(10)[::-1])
                w /= w.sum()
                ewma_val = float(np.dot(w, hist[-10:]))
            elif len(hist) >= 1:
                ewma_val = hist[-1]
            elif global_obs_buf:
                ewma_val = float(np.median(global_obs_buf[-200:]))
            else:
                ewma_val = np.nan

            if src_df.loc[row_i, 'on_expiry']:
                # Expiry day: spline only; prior-day EWMA is irrelevant under gamma explosion
                filled = sp_est if pd.notna(sp_est) else (ewma_val if pd.notna(ewma_val) else 0.22)
            else:
                n_obs_bar = sum(1 for c in opt_cols if pd.notna(ts[c]))
                spline_w  = 0.70 if n_obs_bar < 16 else 0.90
                if pd.notna(sp_est) and pd.notna(ewma_val):
                    filled = spline_w * sp_est + (1 - spline_w) * ewma_val
                elif pd.notna(sp_est):
                    filled = sp_est
                elif pd.notna(ewma_val):
                    filled = ewma_val
                else:
                    filled = 0.22

            filled = max(0.005, filled)
            stage1_df.loc[row_i, col] = filled
            spline_store[(row_i, col)] = (sp_est if pd.notna(sp_est) else filled, 1)
        else:
            spline_store[(row_i, col)] = (sp_est if pd.notna(sp_est) else float(raw_val), 0)

    # Append only real observed values (never predictions) to history
    for col in opt_cols:
        v = ts[col]
        if pd.notna(v):
            col_history[col].append(float(v))
            global_obs_buf.append(float(v))

    global_mean_ts.append(
        cur_mean if pd.notna(cur_mean)
        else (global_mean_ts[-1] if global_mean_ts else 0.20)
    )

print("Stage 1 complete.")
print(f"  Gaps filled   : {src_df[opt_cols].isna().sum().sum():,}")
print(f"  Remaining NaN : {stage1_df[opt_cols].isna().sum().sum()}")

Stage 1 complete.
  Gaps filled   : 5,460
  Remaining NaN : 0


## 8. Feature Engineering

47 features total, organised into:
- **Structural:** moneyness, strike rank, DTE, time of day
- **Surface shape:** ATM level, skew, dispersion, curvature, wing ratio
- **Temporal:** lagged IVs, rolling means, velocity
- **Interactions:** moneyness × spline confidence, DTE × moneyness, etc.

The interaction features were motivated by the observation that spline errors correlate with both moneyness *and* how many strikes were observed.

In [9]:
# Feature registry — order matters for reproducibility
BASE_FEATS = [
    'log_moneyness', 'moneyness', 'std_moneyness', 'is_call', 'strike_rank',
    'dte', 'time_index', 'time_frac', 'day_index', 'is_early_day',
    'spot', 'global_atm_iv', 'atm_ce', 'atm_pe', 'atm_iv_level',
    'mean_ce_iv', 'mean_pe_iv', 'min_ce_iv', 'min_pe_iv',
    'n_obs_ce', 'n_obs_pe', 'n_obs_ratio_ce', 'n_obs_ratio_pe',
    'spline_confidence', 'skew_iv', 'surf_disp', 'ce_curv', 'pe_curv',
    'wing_ratio', 'boundary_gap',
    'iv_lag1', 'iv_lag2', 'iv_roll3', 'iv_roll10', 'iv_velocity',
    'iv_lag1_filled', 'iv_roll3_filled', 'iv_roll10_filled',
    'global_iv_lag1', 'spline_pred',
    'spot_ret1', 'ret1_x_call',
]
INTERACT_FEATS = [
    'lag1_spline_ratio', 'lm_x_spline_conf', 'lm_x_n_obs',
    'surf_disp_x_rank', 'dte_x_lm',
]
ALL_FEATS = list(dict.fromkeys(BASE_FEATS + INTERACT_FEATS))
print(f"Feature count: {len(ALL_FEATS)}  ({len(BASE_FEATS)} base + {len(INTERACT_FEATS)} interaction)")

Feature count: 47  (42 base + 5 interaction)


## 9. Long-Format Reshaping

Convert from wide format (one row per timestamp, 28 IV columns) to long format (one row per timestamp × strike pair). This is the format gradient boosting needs — each row is an independent prediction target.

In [10]:
# Reshape wide DataFrame into long format: one row per (timestamp, strike) pair.
# This is the format required by the gradient boosting models.

record_buf = []

for idx, ts in src_df.iterrows():
    spot_px  = float(ts['underlying_price'])
    ss       = surface_stats(ts, spot_px)
    atm_lvl  = ss['atm_iv_level'] if pd.notna(ss['atm_iv_level']) else 0.20
    tau_yrs  = max(0.5, float(ts['days_left'])) / 365.0
    g_lag1   = global_mean_ts[idx - 1] if idx > 0 else 0.20
    n_ce_now = len([c for c in call_cols if pd.notna(ts[c])])
    n_pe_now = len([c for c in put_cols  if pd.notna(ts[c])])

    for col in opt_cols:
        k       = col_meta[col]['strike']
        is_call = 1 if col_meta[col]['type'] == 'CE' else 0
        lm      = np.log(k / spot_px)
        sp_val  = spline_store.get((idx, col), (np.nan, 0))[0]

        if is_call:
            s_rank = call_strikes.index(k) / max(1, len(call_strikes) - 1)
            n_same = n_ce_now
        else:
            s_rank = put_strikes.index(k)  / max(1, len(put_strikes)  - 1)
            n_same = n_pe_now

        record_buf.append({
            'row_idx': idx, 'col': col,
            'iv':         float(ts[col]) if pd.notna(ts[col]) else np.nan,
            'is_missing': int(pd.isna(ts[col])),
            'is_call': is_call, 'strike': k, 'spot': spot_px,
            'log_moneyness':  lm,
            'moneyness':      k / spot_px,
            'std_moneyness':  lm / (atm_lvl * np.sqrt(tau_yrs)),
            'strike_rank':    s_rank,
            'dte':            int(ts['days_left']),
            'time_index':     int(ts['bar_num']),
            'time_frac':      float(ts['session_pct']),
            'day_index':      int(ts['day_seq']),
            'is_early_day':   int(ts['day_seq'] <= 1),
            'is_expiry':      int(ts['on_expiry']),
            'spline_pred':    sp_val,
            'global_iv_lag1': g_lag1,
            'n_obs_ratio_ce': n_ce_now / 14.0,
            'n_obs_ratio_pe': n_pe_now / 14.0,
            'spline_confidence': n_same / 14.0,
            'spot_ret1': float(ts['spot_chg']),
            **ss,
        })

long_tbl = pd.DataFrame(record_buf).sort_values(['col', 'row_idx']).reset_index(drop=True)
print(f"Long table shape: {long_tbl.shape}  ({len(src_df)} timestamps x {len(opt_cols)} strikes)")
long_tbl[['row_idx', 'col', 'iv', 'is_missing', 'log_moneyness', 'spline_pred']].head(4)

Long table shape: (27300, 39)  (975 timestamps x 28 strikes)


,row_idx,col,iv,is_missing,log_moneyness,spline_pred
0,0,NIFTY27JAN2623800PE,0.17840,0,-0.092696,0.175096
1,1,NIFTY27JAN2623800PE,0.17962,0,-0.093835,0.179414
2,2,NIFTY27JAN2623800PE,0.18010,0,-0.093756,0.179669
3,3,NIFTY27JAN2623800PE,0.18174,0,-0.093358,0.185790


## 10. Temporal Lag Features

IV is highly autocorrelated — the best predictor of the next bar's IV is the current bar's IV. I compute:
- `iv_lag1`, `iv_lag2`: previous observed values for this strike
- `iv_roll3`, `iv_roll10`: rolling means over 3 and 10 bars
- `iv_velocity`: first difference (momentum signal)

**Important:** I use a strict read-before-append pattern to avoid lookahead bias. The history buffer is only updated *after* reading from it.

In [11]:
# Temporal lag features — IV is highly autocorrelated across bars.
# Read-before-append pattern: the history buffer is read FIRST, THEN the
# current value is appended. This guarantees iv_lag1 at bar T contains
# only the IV from bar T-1 (strictly no lookahead).

for feat in ['iv_lag1', 'iv_lag2', 'iv_roll3', 'iv_roll10', 'iv_velocity']:
    long_tbl[feat] = np.nan

for col_name, col_grp in long_tbl.groupby('col'):
    for d_idx, day_grp in col_grp.groupby('day_index'):
        positions = day_grp.index.tolist()
        iv_seq    = day_grp['iv'].values.copy()
        n_bars    = len(iv_seq)

        l1  = np.full(n_bars, np.nan)
        l2  = np.full(n_bars, np.nan)
        r3  = np.full(n_bars, np.nan)
        r10 = np.full(n_bars, np.nan)
        vel = np.full(n_bars, np.nan)
        buf = []

        for bar_i, iv_val in enumerate(iv_seq):
            # --- READ phase (past only) ---
            if len(buf) >= 1:
                l1[bar_i]  = buf[-1]
                r3[bar_i]  = np.mean(buf[-3:])
                r10[bar_i] = np.mean(buf[-10:])
            if len(buf) >= 2:
                l2[bar_i]  = buf[-2]
                vel[bar_i] = buf[-1] - buf[-2]
            # --- APPEND phase (current enters future bars only) ---
            if not np.isnan(iv_val):
                buf.append(iv_val)

        long_tbl.loc[positions, 'iv_lag1']     = l1
        long_tbl.loc[positions, 'iv_lag2']     = l2
        long_tbl.loc[positions, 'iv_roll3']    = r3
        long_tbl.loc[positions, 'iv_roll10']   = r10
        long_tbl.loc[positions, 'iv_velocity'] = vel

# Opening-bar NaNs: use spline as a financially grounded proxy for missing history
long_tbl['iv_lag1_filled']   = long_tbl['iv_lag1'].fillna(long_tbl['spline_pred'])
long_tbl['iv_roll3_filled']  = long_tbl['iv_roll3'].fillna(long_tbl['spline_pred'])
long_tbl['iv_roll10_filled'] = long_tbl['iv_roll10'].fillna(long_tbl['spline_pred'])

# Interaction terms
long_tbl['lag1_spline_ratio'] = long_tbl['iv_lag1_filled'] / long_tbl['spline_pred'].clip(0.01)
long_tbl['lm_x_spline_conf']  = long_tbl['log_moneyness'].abs() * (1 - long_tbl['spline_confidence'])
long_tbl['lm_x_n_obs']        = long_tbl['log_moneyness'].abs() * np.where(
    long_tbl['is_call'] == 1, long_tbl['n_obs_ce'], long_tbl['n_obs_pe'])
long_tbl['surf_disp_x_rank']  = long_tbl['surf_disp'] * long_tbl['strike_rank']
long_tbl['dte_x_lm']          = np.log(long_tbl['dte'].clip(1)) * long_tbl['log_moneyness'].abs()
long_tbl['ret1_x_call']       = long_tbl['spot_ret1'].fillna(0) * long_tbl['is_call']

print("Lag and interaction features ready.")
print(f"  iv_lag1 populated    : {long_tbl['iv_lag1'].notna().sum():,} / {len(long_tbl):,}")
print(f"  iv_lag1_filled nulls : {long_tbl['iv_lag1_filled'].isna().sum()}")

Lag and interaction features ready.
  iv_lag1 populated    : 26,838 / 27,300
  iv_lag1_filled nulls : 0


## 11. Model Hyperparameters

Two models per sub-problem (CE and PE), each with different inductive biases:
- **LightGBM:** leaf-wise growth, fast, good with continuous features
- **CatBoost:** symmetric (oblivious) trees, strong regularisation


The expiry model uses fewer leaves and stronger regularisation because the training set is much smaller (1,350 rows vs 10,000+).

In [12]:
# Hyperparameter blocks — one per sub-problem.
# CE and PE share the same architecture but are trained on separate datasets.
# Expiry model is more regularised (fewer leaves, lower LR, higher penalties)
# because gamma-regime data is scarce and highly non-stationary.

lgbm_cfg_ce = dict(
    n_estimators=5000, learning_rate=0.002, num_leaves=31,
    min_child_samples=40, subsample=0.75, colsample_bytree=0.70,
    reg_alpha=0.5, reg_lambda=1.0, random_state=42,
    objective='regression', verbose=-1
)
lgbm_cfg_pe = dict(
    n_estimators=5000, learning_rate=0.002, num_leaves=31,
    min_child_samples=40, subsample=0.75, colsample_bytree=0.70,
    reg_alpha=0.5, reg_lambda=1.0, random_state=42,
    objective='regression', verbose=-1
)
lgbm_cfg_exp = dict(
    n_estimators=8000, learning_rate=0.001, num_leaves=7,
    min_child_samples=30, subsample=0.70, colsample_bytree=0.60,
    reg_alpha=1.5, reg_lambda=4.0, random_state=42,
    objective='regression', verbose=-1
)
catb_cfg = dict(
    iterations=3000, learning_rate=0.02, depth=6,
    subsample=0.75, colsample_bylevel=0.70, reg_lambda=1.0,
    random_seed=42, verbose=0, loss_function='RMSE'
)

print("Hyperparameter configs ready.")

Hyperparameter configs ready.


## 12. Stage 2 — Training the Ensemble

Each model targets the residual (observed IV − spline prediction), not the raw IV. This keeps the target distribution tight and well-behaved. The ensemble blend is:

$$\text{correction} = 0.50 \times \text{LGB} + 0.50 \times \text{CB}$$

The 50/50 weighting gives both models equal influence because it consistently had lower validation error, while CatBoost provides diversity through its symmetric tree architecture.

In [13]:
# Train three models per sub-problem. Each model sees the spline residual as its
# target — not the raw IV. This keeps the learning signal clean and interpretable.

# CE calls — full-data training (only 13 days available; holdout wastes signal)
ce_train_set = long_tbl[
    (long_tbl['is_expiry']  == 0) &
    (long_tbl['is_call']    == 1) &
    (long_tbl['is_missing'] == 0) &
    long_tbl['iv'].notna()
].copy()
ce_train_set['err'] = ce_train_set['iv'] - ce_train_set['spline_pred']
ce_feat_med = ce_train_set[ALL_FEATS].median()
X_ce = ce_train_set[ALL_FEATS].fillna(ce_feat_med)
y_ce = ce_train_set['err'].values

lgbm_ce = lgb.LGBMRegressor(**lgbm_cfg_ce);  lgbm_ce.fit(X_ce, y_ce)
catb_ce  = CatBoostRegressor(**catb_cfg);      catb_ce.fit(X_ce, y_ce)
print(f"CE  models trained on {len(X_ce):,} rows")

# PE puts
pe_train_set = long_tbl[
    (long_tbl['is_expiry']  == 0) &
    (long_tbl['is_call']    == 0) &
    (long_tbl['is_missing'] == 0) &
    long_tbl['iv'].notna()
].copy()
pe_train_set['err'] = pe_train_set['iv'] - pe_train_set['spline_pred']
pe_feat_med = pe_train_set[ALL_FEATS].median()
X_pe = pe_train_set[ALL_FEATS].fillna(pe_feat_med)
y_pe = pe_train_set['err'].values

lgbm_pe = lgb.LGBMRegressor(**lgbm_cfg_pe);  lgbm_pe.fit(X_pe, y_pe)
catb_pe  = CatBoostRegressor(**catb_cfg);      catb_pe.fit(X_pe, y_pe)
print(f"PE  models trained on {len(X_pe):,} rows")

# Expiry day — regularised LightGBM with early stopping
exp_train_set = long_tbl[
    (long_tbl['is_expiry']  == 1) &
    (long_tbl['is_missing'] == 0) &
    long_tbl['iv'].notna()
].copy().reset_index(drop=True)
exp_train_set['err'] = exp_train_set['iv'] - exp_train_set['spline_pred']

hold_mask    = (exp_train_set.index % 5 == 0)
exp_feat_med = exp_train_set.loc[~hold_mask, ALL_FEATS].median()
X_exp_tr  = exp_train_set.loc[~hold_mask, ALL_FEATS].fillna(exp_feat_med)
y_exp_tr  = exp_train_set.loc[~hold_mask, 'err'].values
X_exp_val = exp_train_set.loc[hold_mask,  ALL_FEATS].fillna(exp_feat_med)
y_exp_val = exp_train_set.loc[hold_mask,  'err'].values

lgbm_exp = lgb.LGBMRegressor(**lgbm_cfg_exp)
lgbm_exp.fit(X_exp_tr, y_exp_tr,
             eval_set=[(X_exp_val, y_exp_val)],
             callbacks=[lgb.early_stopping(150, verbose=False),
                        lgb.log_evaluation(9999)])
print(f"Expiry model  : best_iter={lgbm_exp.best_iteration_:,}  train_rows={len(X_exp_tr):,}")

CE  models trained on 10,104 rows
PE  models trained on 10,044 rows
Expiry model  : best_iter=8,000  train_rows=1,353


## 13. Stage 2 — Applying Corrections

The ensemble correction is scaled by a zone-specific alpha before being added to the spline baseline:
- **Near ATM (|log-m| < 0.02):** α = 0.70 — spline is already quite accurate here
- **Moderate OTM (0.02 ≤ |log-m| < 0.05):** α = 0.90 — spline struggles most here, trust ML more
- **Deep OTM (|log-m| ≥ 0.05):** α = 0.70 — less data, be conservative

For puts, a uniform α = 0.60 works because put residuals are much smaller.
For expiry day, α = 0.45 — slightly higher than conservative default since the ensemble still captures some gamma structure.

In [14]:
# Apply ensemble corrections to the Stage 1 baseline.
#
# Final formula per cell:
#   CE: spline_pred + alpha(zone) * (0.65*LGB + 0.35*CB)
#   PE: spline_pred + alpha(zone) * (0.65*LGB + 0.35*CB)
#   Expiry: spline_pred + 0.40 * LGB   (single model, conservative)

result_df  = stage1_df.copy()
needs_fill = long_tbl['is_missing'] == 1
is_expiry  = long_tbl['is_expiry']  == 1
is_call    = long_tbl['is_call']    == 1

# CE calls on normal trading days
ce_pred_rows = long_tbl[~is_expiry & needs_fill & is_call]
if len(ce_pred_rows) > 0:
    X_inf  = ce_pred_rows[ALL_FEATS].fillna(ce_feat_med)
    ens_ce = (0.50 * lgbm_ce.predict(X_inf) +
              0.50 * catb_ce.predict(X_inf))
    lm_abs    = ce_pred_rows['log_moneyness'].abs().values
    alpha_arr = np.where(lm_abs < 0.02, CE_ALPHA_ATM,
                np.where(lm_abs < 0.05, CE_ALPHA_MOD, CE_ALPHA_WING))
    final_ce  = np.clip(ce_pred_rows['spline_pred'].values + alpha_arr * ens_ce, 0.005, None)
    for (ridx, col), val in zip(zip(ce_pred_rows['row_idx'], ce_pred_rows['col']), final_ce):
        result_df.at[int(ridx), col] = val

# PE puts on normal trading days
pe_pred_rows = long_tbl[~is_expiry & needs_fill & ~is_call]
if len(pe_pred_rows) > 0:
    X_inf  = pe_pred_rows[ALL_FEATS].fillna(pe_feat_med)
    ens_pe = (0.50 * lgbm_pe.predict(X_inf) +
              0.50 * catb_pe.predict(X_inf))
    lm_abs    = pe_pred_rows['log_moneyness'].abs().values
    alpha_arr = np.where(lm_abs < 0.02, PE_ALPHA_ATM,
                np.where(lm_abs < 0.05, PE_ALPHA_MOD, PE_ALPHA_WING))
    final_pe  = np.clip(pe_pred_rows['spline_pred'].values + alpha_arr * ens_pe, 0.005, None)
    for (ridx, col), val in zip(zip(pe_pred_rows['row_idx'], pe_pred_rows['col']), final_pe):
        result_df.at[int(ridx), col] = val

# Expiry day — single LightGBM with conservative alpha
exp_pred_rows = long_tbl[is_expiry & needs_fill]
if len(exp_pred_rows) > 0:
    X_inf    = exp_pred_rows[ALL_FEATS].fillna(exp_feat_med)
    exp_preds = lgbm_exp.predict(X_inf)
    final_ex  = np.clip(exp_pred_rows['spline_pred'].values + EXPIRY_ALPHA * exp_preds, 0.005, None)
    for (ridx, col), val in zip(zip(exp_pred_rows['row_idx'], exp_pred_rows['col']), final_ex):
        result_df.at[int(ridx), col] = val

# Safety net — should not trigger under normal execution
for col in opt_cols:
    if result_df[col].isna().any():
        fb = src_df[col].median()
        result_df[col] = result_df[col].fillna(fb if pd.notna(fb) else 0.22)

total_preds = len(ce_pred_rows) + len(pe_pred_rows) + len(exp_pred_rows)
print(f"CE: {len(ce_pred_rows):,}  |  PE: {len(pe_pred_rows):,}  |  Expiry: {len(exp_pred_rows):,}")
print(f"Total predicted : {total_preds:,}  (expected 5460)")
print(f"Remaining NaN   : {result_df[opt_cols].isna().sum().sum()}")
print(f"All positive    : {(result_df[opt_cols] > 0).all().all()}")

CE: 2,496  |  PE: 2,556  |  Expiry: 408
Total predicted : 5,460  (expected 5460)
Remaining NaN   : 0
All positive    : True


## 14. Export Submission

Restore the original CSV row order, extract only the cells that were missing in the input, and save.

In [15]:
# Export submission file.
# Restore the original CSV row order, melt to long format, keep only missing cells.

result_df['orig_pos'] = src_df['orig_pos'].values
result_df = (
    result_df
    .sort_values('orig_pos')
    .drop(columns=['orig_pos', 'cal_date', 'days_left', 'bar_num',
                   'session_pct', 'day_seq', 'on_expiry'], errors='ignore')
    .reset_index(drop=True)
)
result_df['datetime'] = raw_src['datetime'].values

melted_preds = result_df.melt(id_vars=['datetime'], value_vars=opt_cols,
                               var_name='col', value_name='predicted')
melted_orig  = raw_src[['datetime'] + opt_cols].melt(
    id_vars=['datetime'], value_vars=opt_cols,
    var_name='col', value_name='original')
melted_preds['original'] = melted_orig['original']

submission = melted_preds[melted_preds['original'].isna()].copy()
submission['id'] = submission['datetime'] + ID_SEP + submission['col']
submission = (submission[['id', 'predicted']]
              .rename(columns={'predicted': 'value'})
              .sort_values('id')
              .reset_index(drop=True))
submission.to_csv(OUT_PATH, index=False)

print(f"Output        : {OUT_PATH}")
print(f"Row count     : {len(submission)}  (expected 5460)")
print(f"Value range   : {submission['value'].min():.4f} - {submission['value'].max():.4f}")
print(f"All positive  : {(submission['value'] > 0).all()}")
print(f"All below 10  : {(submission['value'] < 10).all()}")
print(f"No NaN        : {submission['value'].notna().all()}")
submission.head(8)

Output        : submission.csv
Row count     : 5460  (expected 5460)
Value range   : 0.0355 - 5.8082
All positive  : True
All below 10  : True
No NaN        : True


,id,value
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163743
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.114391
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.101220
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.170289
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.159831
5,07-01-2026 09:20||NIFTY27JAN2624800PE,0.128134
6,07-01-2026 09:20||NIFTY27JAN2625000PE,0.118581
7,07-01-2026 09:20||NIFTY27JAN2625300CE,0.105447
